# 第 8 章 · 动态工作流进阶：恢复与运行保障

> 第七章回答“如何安排步骤”；本章回答“暂停、失败或重启后，怎样继续并交付正确结果”。先做小实验，再用一个完整内容审核工程串起所有机制。

**本章目标**：
1. 验证人工确认的接受/拒绝与父节点重入；
2. 认识 invocation、execution ID、重试、超时和外部幂等的边界；
3. 实际运行“批量写作 → 规则检查 → 人工确认 → 持久归档”，在三个独立进程间完成故障恢复。

按顺序运行本章，**无需复用第七章的内存变量**。环境与启动方式相同：`uv sync` 后用项目虚拟环境启动 Jupyter，并配置 `DEEPSEEK_API_KEY`。锁定版 ADK 2.6.3；异步 SQLite 所需依赖已纳入项目。

综合案例输入两项，每项最多修改一次，正常业务路径最多 **4 次模型调用步骤**；恢复不得重复请求已完成模型步骤。其余实验通过真实 ADK、SQLite 和 Artifact API 执行，故障为可控注入。章末汇总本次实际执行证据。

**怎么读这一章**：前半部分（第 2–4 节）是几个**独立的小实验**，每个只验证一件事，可以单独运行、单独理解；后半部分（第 5 节）再把它们组装成一个跨三个进程的完整工程。

如果时间有限，**先只跑第 2 节**：把“暂停 → 恢复 → 回放”这一件事弄明白，后面所有内容都建立在它之上。第 2 节看懂之后，第 3、4 节会快很多。

---
## 1. 运行上下文与事件收集

保留与第七章相同的 Runner 骨架，`Event.output` 的根归属用于提取整条流程结果。

> ⚠️ **一个容易踩的坑**：流程返回 `None` 时，**不能**直接判定为“失败”或“被拒绝”。**暂停**（在等待人工输入）同样表现为没有根输出。正确做法是再去检查有没有待处理的请求事件——本章后面的 `pending_input()` 就是做这件事的。

> 📌 本章会反复用到三个词，先在这里对齐：**暂停**（流程停在等待人工输入的位置）、**恢复**（拿到输入后继续往前走）、**回放**（恢复时不重跑已完成的步骤，直接使用 checkpoint 里的旧结果）。读到后面任何一段卡住时，回到这三个词。

In [1]:
import asyncio
import importlib.metadata as metadata
import json
import os
from uuid import uuid4

# 使用随 LiteLLM 安装的成本表，不影响真实模型请求。
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk import Context, Workflow
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import node
from google.genai import types
from pydantic import BaseModel, Field

assert os.getenv("DEEPSEEK_API_KEY"), "请先配置 DEEPSEEK_API_KEY，再执行本章。"
assert metadata.version("google-adk") == "2.6.3", (
    "请从项目根目录执行 uv sync。"
)
APP, USER = "adk_ch08", "student"
checks = {}
print("运行环境：", metadata.version("google-adk"), "；DeepSeek 密钥已配置")

运行环境： 2.6.3 ；DeepSeek 密钥已配置


In [2]:
def input_text(value) -> str:
    if isinstance(value, types.Content):
        return "".join(part.text or "" for part in value.parts or [])
    if isinstance(value, str):
        return value
    raise TypeError(f"入口需要 Content 或 str，实际是 {type(value).__name__}")


async def collect(runner, session_id, message, *, invocation_id=None):
    output, events = None, []
    async for event in runner.run_async(
        user_id=USER,
        session_id=session_id,
        new_message=message,
        invocation_id=invocation_id,
    ):
        events.append(event)
        if event.node_info and event.output is not None:
            if any(
                "/" not in path for path in event.node_info.output_for or []
            ):
                output = event.output
    return output, events


async def start(
    workflow, payload, *, session_service=None, artifact_service=None
):
    service = session_service or InMemorySessionService()
    session_id = uuid4().hex
    await service.create_session(
        app_name=APP, user_id=USER, session_id=session_id
    )
    runner = Runner(
        agent=workflow,
        app_name=APP,
        session_service=service,
        artifact_service=artifact_service,
    )
    text = (
        payload
        if isinstance(payload, str)
        else json.dumps(payload, ensure_ascii=False)
    )
    message = types.Content(role="user", parts=[types.Part(text=text)])
    output, events = await collect(runner, session_id, message)
    return output, events, runner, session_id

## 2. 人工确认：父函数重入，已完成子节点回放

`RequestInput` 发出请求后暂停；恢复回复匹配 function call ID、名称和 invocation。简单输入叶子使用 `rerun_on_resume=False`，调度父节点使用 `True`。

```mermaid
sequenceDiagram
    participant P as 编排父节点
    participant D as 准备节点
    participant H as 确认节点
    participant A as Artifact
    P->>D: 生成候选稿
    D-->>P: 结果写入 checkpoint
    P->>H: RequestInput 后暂停
    Note over P: 恢复时重新进入父函数
    P->>D: 回放已完成结果
    P->>H: 取得匹配的人工回复
    P->>A: 接受才保存
```

下面以计数验证重入和回放；归档真实调用内存 Artifact API，并读回内容。这里的人工决定是测试输入，生产中应来自用户界面或可信业务系统。

In [3]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory

from google.adk.artifacts import InMemoryArtifactService
from google.adk.events import RequestInput


class ApprovalDecision(BaseModel):
    approved: bool = Field(strict=True)


approval_counts = Counter()


@node
def prepare_candidate(node_input: str) -> dict:
    approval_counts["prepare"] += 1
    return {"title": node_input, "body": "候选稿已经生成，等待人工审核。"}


@node(rerun_on_resume=False)
def ask_approval(node_input: dict):
    return RequestInput(
        message="是否接受候选稿？",
        payload=node_input,
        response_schema=ApprovalDecision,
    )


@node
async def save_candidate(ctx: Context, node_input: dict) -> dict:
    version = await ctx.save_artifact(
        "candidate.md", types.Part.from_text(text=node_input["body"])
    )
    approval_counts["archive"] += 1
    return {"status": "archived", "version": version}


@node(rerun_on_resume=True)
async def approval_workflow(ctx: Context, node_input):
    approval_counts["parent"] += 1
    draft = await ctx.run_node(prepare_candidate, input_text(node_input))
    reply = await ctx.run_node(ask_approval, draft)
    if not ApprovalDecision.model_validate(reply).approved:
        return {"status": "rejected"}
    return await ctx.run_node(save_candidate, draft)


approval_app = Workflow(
    name="approval_app", edges=[("START", approval_workflow)]
)

In [4]:
def pending_input(events):
    calls = [
        (event.invocation_id, part.function_call)
        for event in events
        for part in ((event.content.parts or []) if event.content else [])
        if part.function_call
        and part.function_call.name == "adk_request_input"
    ]
    if len(calls) != 1:
        raise ValueError(f"本例需要恰好一个确认请求，实际是 {len(calls)}")
    return calls[0]


async def answer(runner, session_id, events, approved):
    invocation_id, call = pending_input(events)
    message = types.Content(
        role="user",
        parts=[
            types.Part(
                function_response=types.FunctionResponse(
                    id=call.id,
                    name=call.name,
                    response={"approved": approved},
                ),
            )
        ],
    )
    return await collect(
        runner, session_id, message, invocation_id=invocation_id
    )

In [5]:
for decision, expected in [(True, "archived"), (False, "rejected")]:
    approval_counts.clear()
    artifacts = InMemoryArtifactService()
    paused, events, runner, session_id = await start(
        approval_app,
        "工作流介绍",
        artifact_service=artifacts,
    )
    assert paused is None and approval_counts == {"parent": 1, "prepare": 1}
    result, _ = await answer(runner, session_id, events, decision)
    assert result["status"] == expected
    assert approval_counts["parent"] == 2 and approval_counts["prepare"] == 1
    assert approval_counts["archive"] == int(decision)
    saved = await artifacts.load_artifact(
        app_name=APP,
        user_id=USER,
        session_id=session_id,
        filename="candidate.md",
    )
    assert (saved is not None) is decision
    print(expected, dict(approval_counts))
checks["接受/拒绝、父节点重入与真实 Artifact"] = True

archived {'parent': 2, 'prepare': 1, 'archive': 1}
rejected {'parent': 2, 'prepare': 1}


> 📌 **这是本章最重要的一句话**：Python 的调用栈和局部变量**没有被原样冻结**。恢复时，父函数从函数头重新跑一遍；已经完成的子节点则由 checkpoint 把旧结果**回放**给你。
>
> **这不是框架偷懒，而是第 7 章那个定义的直接结果。** 那里说过：动态工作流的“路径”就是 Python 程序本身（第 7 章 §1.1）。而解释器**没法从函数体中间某一行接着执行**——所以除了“重进函数 + 回放子节点”，框架别无选择。**“动态”正是本章存在的理由。**

换个比喻：checkpoint 存的是**每一步的答案**，不是**你算到哪儿的草稿纸**。所以重新发卷子时，之前答对的题会直接印上原来的答案——而你要做的，只是接着往下答。

这个性质有两个直接后果：① 父函数里的普通代码必须"跑多少遍结果都一样"；② 真正的副作用必须放进**独立节点**，并且另有业务保障。

---

## 3. Execution ID：改变顺序后验证任务匹配

恢复能不能对上号，取决于**身份**能不能对上。这里有三个层次的 ID，先把它们分清楚：

| ID | 定位什么 | 谁生成 |
|---|---|---|
| `session_id` | 一次会话（一个任务的全过程） | 创建会话时由你指定 |
| `invocation_id` | 会话中的**一次执行** | Runner |
| `run_id` / `node_info.path` | 这次执行中的**一步** | 框架按节点名和调用计数生成 |

**默认 ID 依赖节点名和调用计数**，所以通常的做法是：把任务快照固定下来，安排顺序也固定下来。但现实里顺序是会变的（排序规则调整、数据补齐、并发完成次序不同），这时默认 ID 就对不上号了。

下面刻意**改变同一快照的处理顺序**，改用业务 ID 映射 `run_id`。恢复后逐项核对结果和实际调用次数，证明这不是"只把路径名字改漂亮"。

> ⚠️ 自定义 `run_id` 要谨慎：**同一个 ID 在恢复时不能代表不同的输入**。否则 checkpoint 会把 A 的结果当成 B 的答案还给你——而且**不会报错**。

In [6]:
from collections import Counter

item_calls = Counter()
item_order = "forward"


@node
def process_item(node_input: dict) -> dict:
    item_calls[node_input["id"]] += 1
    return {"id": node_input["id"], "label": node_input["topic"].upper()}


@node(rerun_on_resume=True)
async def stable_batch(ctx: Context, node_input):
    items = json.loads(input_text(node_input))
    ids = [item["id"] for item in items]
    if len(ids) != len(set(ids)):
        raise ValueError("业务 ID 必须唯一")
    ordered = sorted(
        items, key=lambda item: item["id"], reverse=item_order == "reverse"
    )
    results = await asyncio.gather(
        *[
            ctx.run_node(process_item, item, run_id=f"item-{item['id']}")
            for item in ordered
        ]
    )
    decision = await ctx.run_node(ask_approval, {"items": results})
    return {
        "approved": ApprovalDecision.model_validate(decision).approved,
        "items": results,
    }


stable = Workflow(name="stable", edges=[("START", stable_batch)])
item_calls.clear()
item_order = "forward"
paused, stable_events, stable_runner, stable_session = await start(
    stable,
    [
        {"id": "a", "topic": "adk"},
        {"id": "b", "topic": "workflow"},
    ],
)
assert paused is None and item_calls == {"a": 1, "b": 1}
item_order = "reverse"
result, _ = await answer(stable_runner, stable_session, stable_events, True)
assert result["items"] == [
    {"id": "b", "label": "WORKFLOW"},
    {"id": "a", "label": "ADK"},
]
assert item_calls == {"a": 1, "b": 1}
checks["顺序变化后的 ID 匹配与子节点回放"] = True
print("恢复后的结果：", result, "；实际执行次数：", dict(item_calls))

恢复后的结果： {'approved': True, 'items': [{'id': 'b', 'label': 'WORKFLOW'}, {'id': 'a', 'label': 'ADK'}]} ；实际执行次数： {'a': 1, 'b': 1}


## 4. 重试、超时与副作用：分别控制预算

这一节讨论三件容易被混为一谈的事：**重试**（失败了要不要再来一次）、**超时**（等多久算失败）、**副作用**（外部世界已经被改了怎么办）。它们要分开控制。

### 4.0 先搞清楚默认值：不写 `retry_config` 反而不会重试

这是本章最容易记错的一个点，先把它钉死：

| 写法 | 总尝试次数 | 说明 |
|---|---|---|
| **不写** `retry_config` | **1 次** | 失败即失败，**框架不重试** |
| `RetryConfig()` | **5 次** | 只写 `RetryConfig()` 而不写 `max_attempts`，会**静默拿到 5 次预算** |
| `RetryConfig(max_attempts=1)` | 1 次 | 显式声明"不重试" |
| `RetryConfig(max_attempts=2)` | 2 次 | **包含**第一次请求，所以只额外重试 1 次 |

其余几个默认值：`initial_delay=1.0` 秒、`max_delay=60.0` 秒、`backoff_factor=2.0`、`jitter=1.0`。也就是默认的等待节奏是 1s → 2s → 4s → 8s。

> ⚠️ 真正的陷阱在第二行：**你以为"不写 `max_attempts` 就是只重试 1 次"，实际上是 5 次。** 对一个会写外部存储的节点来说，这意味着同一个副作用最多可能被执行 5 遍。

下面用实验把这个默认值**数出来**——不靠文档，靠 `ctx.attempt_count`（从 1 开始的尝试序号）。

In [7]:
# 4.0 实验：只写 RetryConfig() 而不写 max_attempts，数出框架的默认预算
from google.adk.workflow import RetryConfig

default_budget_attempts = []

@node(
    retry_config=RetryConfig(
        # 故意不写 max_attempts：这里就是要用框架的默认值
        # 只为让实验跑得快才把等待压到 0.01 秒；默认是 1 秒起步、每次翻倍
        initial_delay=0.01,
        max_delay=0.01,
        jitter=0,
        exceptions=[ConnectionError],
    )
)
def always_failing_write(ctx: Context, node_input: dict) -> dict:
    default_budget_attempts.append(ctx.attempt_count)
    raise ConnectionError("故障注入：每次必失败，用来数默认重试次数")

@node(rerun_on_resume=True)
async def default_budget_entry(ctx: Context, node_input):
    try:
        return await ctx.run_node(
            always_failing_write, json.loads(input_text(node_input))
        )
    except Exception as error:
        return {
            "failed_with": type(error).__name__,
            "attempts": list(default_budget_attempts),
        }

default_budget_app = Workflow(
    name="default_budget", edges=[("START", default_budget_entry)]
)

# 这 5 次失败都是**预期内**的故障。框架默认会把每次异常连同堆栈打进日志，
# 5 段堆栈会把实验结论淹没 —— 所以这里把 ADK 的日志级别临时压到 CRITICAL。
# （下面 4.1 节的真实重试实验用的是同一招。）
import logging

adk_logger = logging.getLogger("google_adk")
previous_level = adk_logger.level
default_budget_attempts.clear()
try:
    adk_logger.setLevel(logging.CRITICAL)
    budget_result, *_ = await start(default_budget_app, {"key": "budget"})
finally:
    adk_logger.setLevel(previous_level)

# attempt_count 从 1 开始计数，所以"一共试了几次"就是列表的长度
assert len(budget_result["attempts"]) == 5, budget_result
checks["未声明 max_attempts 时的默认重试预算"] = True
print(
    "实际尝试次数：",
    budget_result["attempts"],
    "→ 共",
    len(budget_result["attempts"]),
    "次；最终异常：",
    budget_result["failed_with"],
)


实际尝试次数： [1, 2, 3, 4, 5] → 共 5 次；最终异常： DynamicNodeFailError


（运行这段时，框架默认会把 5 次失败连同堆栈打进日志。那属于**预期内的故障**，不该淹没实验结论——所以实验里把 ADK 的日志级别临时压到了 `CRITICAL`。4.1 节的真实重试实验用的是同一招：**把预期故障当数据展示，而不是当异常刷屏。**）

实验结果会打印出 5 次尝试。请把它和下面这句话一起记住：

> 📌 **"写了 `RetryConfig` 却不写 `max_attempts`"是一个静默的陷阱。** 如果你确实只想试一次，就明确写 `RetryConfig(max_attempts=1)`；如果确实想要重试，就明确写出次数。**默认值不该替你决定重试几次。**

### 4.1 外部写入成功、节点结果尚未记录

上面是"框架会重试几次"。现在换一个更麻烦的问题：**重试的时候，副作用已经发生了怎么办？**

节点真实提交 SQLite 事务后注入连接异常。`RetryConfig(max_attempts=2)` 包含第一次请求，框架只对**指定的异常**重试；业务主键保证两次尝试只产生一条记录。

> ⚠️ 这**不是** checkpoint 自动提供了 exactly-once。是**业务主键**在起作用——"同一个 key 只写一行"这个约束来自你的表结构，不来自框架。

另外注意：**重试次数在本版 ADK 中不会跨恢复累积**。一次执行里试了 2 次，恢复之后又是新的 2 次。长期任务的总预算要你**自己**控制。

In [8]:
import logging
import sqlite3
from contextlib import closing

from google.adk.workflow import NodeTimeoutError, RetryConfig

effect_attempts = []


@node(
    retry_config=RetryConfig(
        max_attempts=2,
        initial_delay=0.01,
        max_delay=0.01,
        jitter=0,
        exceptions=[ConnectionError],
    )
)
def write_record(ctx: Context, node_input: dict) -> dict:
    effect_attempts.append(ctx.attempt_count)
    with closing(sqlite3.connect(effect_database)) as db, db:
        db.execute(
            "INSERT INTO records VALUES (?, ?) ON CONFLICT(key) DO NOTHING",
            (node_input["key"], node_input["payload"]),
        )
        stored = db.execute(
            "SELECT payload FROM records WHERE key=?", (node_input["key"],)
        ).fetchone()[0]
        if stored != node_input["payload"]:
            raise ValueError("同一幂等键不能代表不同内容")
    if ctx.attempt_count == 1:
        raise ConnectionError("故障注入：外部事务已提交，节点结果尚未保存")
    return {"status": "stored", "key": node_input["key"]}


@node(rerun_on_resume=True)
async def effect_entry(ctx: Context, node_input):
    return await ctx.run_node(write_record, json.loads(input_text(node_input)))


effect_app = Workflow(name="effect_app", edges=[("START", effect_entry)])
adk_logger = logging.getLogger("google_adk")
previous_level = adk_logger.level
try:
    # 预期故障改用 Event.error_code 展示，避免整段异常堆栈淹没教学输出。
    adk_logger.setLevel(logging.CRITICAL)
    with TemporaryDirectory(prefix="adk_ch07_effect_") as folder:
        effect_database = str(Path(folder) / "external.db")
        with closing(sqlite3.connect(effect_database)) as db, db:
            db.execute(
                "CREATE TABLE records (key TEXT PRIMARY KEY, payload TEXT)"
            )
        effect_attempts.clear()
        result, effect_events, *_ = await start(
            effect_app,
            {
                "key": "article-42:archive:v1",
                "payload": "已审核文章",
            },
        )
        with closing(sqlite3.connect(effect_database)) as db, db:
            rows = db.execute("SELECT key, payload FROM records").fetchall()
finally:
    adk_logger.setLevel(previous_level)
assert effect_attempts == [1, 2]
assert rows == [("article-42:archive:v1", "已审核文章")]
assert [e.error_code for e in effect_events if e.error_code] == [
    "ConnectionError"
]
assert result["status"] == "stored"
checks["写入后故障、真实重试与业务幂等"] = True
print("实际尝试：", effect_attempts, "；真实存储记录：", rows)
print(
    "预期故障事件：",
    [(e.error_code, e.error_message) for e in effect_events if e.error_code],
)

实际尝试：

 [1, 2] ；真实存储记录： [('article-42:archive:v1', '已审核文章')]
预期故障事件： [('ConnectionError', '故障注入：外部事务已提交，节点结果尚未保存')]


### 4.2 超时与明确降级

真实运行一个超出节点预算的异步等待。处理原则只有一句话：**只有根因确实是超时，才降级；其他异常继续往上抛。**

> ⚠️ 超时**不能撤销已经提交的外部写入**。如果节点在超时前已经把数据写进了数据库，超时只是让这个节点"失败"，外部世界的改动还在。**所以超时和幂等是两件事，不能互相替代。**

> 📌 另外，节点超时**不是整条流程的 SLA**。它只保证"这一步不会无限等下去"，不保证"整个任务能在多久内完成"——后者要靠你自己的总预算。

In [9]:
@node(timeout=0.01, retry_config=RetryConfig(max_attempts=1))
async def slow_lookup(node_input: str) -> str:
    await asyncio.sleep(0.1)
    return "远程结果"


@node(rerun_on_resume=True)
async def timeout_entry(ctx: Context, node_input):
    try:
        return await ctx.run_node(slow_lookup, input_text(node_input))
    except Exception as error:
        if not isinstance(getattr(error, "error", error), NodeTimeoutError):
            raise
        return {"source": "local", "degraded": True}


timeout_app = Workflow(name="timeout_app", edges=[("START", timeout_entry)])
previous_level = adk_logger.level
try:
    adk_logger.setLevel(logging.CRITICAL)
    result, timeout_events, *_ = await start(timeout_app, "读取主题")
finally:
    adk_logger.setLevel(previous_level)
assert result == {"source": "local", "degraded": True}
assert [e.error_code for e in timeout_events if e.error_code] == [
    "NodeTimeoutError"
]
checks["真实节点超时与明确降级"] = True
print(
    result,
    "；预期故障：",
    [e.error_code for e in timeout_events if e.error_code],
)

{'source': 'local', 'degraded': True} ；预期故障： ['NodeTimeoutError']


## 5. 综合提高：可恢复的批量技术内容审核工程

这次把前面的机制放入同一个真实任务：为两份事实说明生成技术介绍，按规则检查并最多修改一次，通过后请求人工确认，最后保存到**文件 Artifact 服务**并登记 SQLite 业务回执。

### 5.1 任务契约与存储分工

| 数据/机制 | 位置 | 为什么 |
|---|---|---|
| 输入快照、子节点结果、人工回复 | `DatabaseSessionService` 的 SQLite | 跨进程回放工作流 |
| 已核对事实、唯一业务 ID、修改预算 | `Job / Item` + 计划叶子节点 | 明确输入与恢复身份 |
| 每篇的通过状态摘要 | 不同的 `ctx.state["article:<id>"]` | 连接第五章状态机制，避免同键覆盖 |
| 完整归档正文与版本 | `FileArtifactService` | 进程退出后仍能读取内容 |
| 执行计数、模型 token、归档回执 | 独立业务 SQLite | 与 checkpoint 独立，验证副作用是否重复 |

**为什么是这几个后端？** ADK 提供了多个会话存储实现，选哪个取决于你要让数据活多久：

| 后端 | 数据活到什么时候 | 适合 | 本章的选择理由 |
|---|---|---|---|
| `InMemorySessionService` | 进程退出即丢 | 单进程实验、单元测试 | 第 7 章用它；**无法跨进程恢复** |
| `SqliteSessionService` | 单机文件，重启还在 | 单机、不想加依赖 | 能力上**够用**，是最小持久方案 |
| `DatabaseSessionService` | 任意 SQLAlchemy 数据库 | 已有数据库、要换 Postgres/MySQL | 本章选它，为了演示**会话存储与业务存储分离** |

> 📌 如果你只是要"重启后还能继续"，`SqliteSessionService` 就够了。本章选 `DatabaseSessionService` 是为了展示**生产形态**——多出来的那层 async 依赖是它的代价，不是必需品。

审批回复除 `approved` 外携带候选稿 SHA-256 摘要；摘要防止回复绑定错稿件，**不替代用户身份认证**。流程版本与内容后端在创建会话时保存，恢复时必须一致。

代码按本仓库 `eval_example` 的方式保存为独立小工程：[workflow.py](workflow_example/workflow.py)、[storage.py](workflow_example/storage.py)、[run.py](workflow_example/run.py)。它们是本章真正执行的源文件；下面展示同一实现的核心代码，再通过 CLI 启动独立进程。

```mermaid
flowchart TD
    I[校验输入并记录计划快照] --> B[最多两个写作工作节点并发]
    B --> R[本地检查 / 最多修改一次]
    R --> Q{全部合格?}
    Q -->|否| N[needs_revision]
    Q -->|是| H[RequestInput / 进程 A 退出]
    H -->|拒绝| D[rejected]
    H -->|接受| A[进程 B 恢复并归档]
    A --> X[注入写入后故障 / 进程 B 退出]
    X --> C[进程 C 恢复]
    C --> F[回放模型结果与审批 / 复用已有归档]
```

In [10]:
import inspect
import subprocess
import sys

from IPython.display import Code, display

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "google-adk").is_dir()
)
PROJECT_DIR = ROOT / "google-adk"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from workflow_example.storage import Ledger
from workflow_example.workflow import (
    FLOW_VERSION,
    Job,
    build_workflow,
    candidate_digest,
    review,
    validate_approval,
)

display(Code(inspect.getsource(build_workflow), language="python"))

def build_workflow(ledger: Ledger, *, mode="deepseek", fail_after_write=False):
    """每个独立进程重建相同节点定义；业务数据从 Session 恢复。"""
    if mode not in {"offline", "deepseek"}:
        raise ValueError("未知内容后端")
    active_workers = 0

    def record_response(callback_context, llm_response):
        usage = llm_response.usage_metadata
        ledger.record_model(
            callback_context.agent_name,
            usage.total_token_count if usage else 0,
        )
        return None

    def model():
        return LiteLlm(
            model="deepseek/deepseek-chat",
            max_tokens=768,
            response_format={"type": "json_object"},
        )

    writer = Agent(
        name="project_writer",
        model=model(),
        mode="single_turn",
        instruction="根据输入 topic 和 notes 写中文技术介绍，遵循 notes 的事实。"
        "正文目标 90～110 个字符，验收 60～150，标点和英文也计数。"
        "正文含 required_terms 中的每个词和‘例如’引出的写作审核例子。"
        "审批用暂停和恢复，不是轮询；checkpoint 不冻结 Python 调用栈。"
        '只输出 JSON：{"title":"标题","body":"正文"}。',
        after_model_callback=record_response,
    )
    reviser = Agent(
        name="project_reviser",
        model=model(),
        mode="single_turn",
        instruction="按 findings 修复 draft，遵循 request.notes。已合格的规则不要破坏。"
        "正文目标 90～110 字符，验收 60～150，包含 required_terms 和‘例如’。"
        "保留事实和写作审核例子，审批通过暂停与恢复实现。"
        '只输出 JSON：{"title":"标题","body":"正文"}。',
        after_model_callback=record_response,
    )

    @node
    def read_plan(node_input) -> dict:
        ledger.count("read_plan")
        text = "".join(part.text or "" for part in node_input.parts or [])
        job = Job.model_validate_json(text)
        return {
            **job.model_dump(),
            "items": [
                item.model_dump()
                for item in sorted(job.items, key=lambda x: x.id)
            ],
        }

    @node
    def offline_writer(node_input: dict) -> dict:
        body = (
            "工作流先由代码安排写作与检查，再按结果决定是否修改。"
            "例如候选稿通过规则后暂停等待人工审核，收到对应回复才继续归档。"
            "过程有预算上限，失败与拒绝都不会被当作成功。"
        )
        if node_input["topic"] == "失败样本":
            body = "太短"
        return {"title": node_input["topic"], "body": body}

    @node(rerun_on_resume=True)
    async def article_worker(ctx: Context, node_input: dict) -> dict:
        nonlocal active_workers
        request = node_input["request"]
        ledger.count(f"article:{request['id']}")
        active_workers += 1
        ledger.record_peak(active_workers)
        try:
            if mode == "deepseek":
                draft = Draft.model_validate_json(
                    await ctx.run_node(writer, request)
                ).model_dump()
            else:
                draft = await ctx.run_node(offline_writer, request)
            for revision in range(node_input["max_revisions"] + 1):
                verdict = review(draft, request)
                if (
                    verdict["passed"]
                    or revision == node_input["max_revisions"]
                ):
                    ctx.state[f"article:{request['id']}"] = {
                        "passed": verdict["passed"],
                        "title": draft["title"],
                    }
                    return {
                        "id": request["id"],
                        "draft": draft,
                        **verdict,
                        "revisions": revision,
                    }
                if mode == "deepseek":
                    draft = Draft.model_validate_json(
                        await ctx.run_node(
                            reviser,
                            {
                                "request": request,
                                "draft": draft,
                                "findings": verdict["findings"],
                            },
                        )
                    ).model_dump()
                else:
                    draft = await ctx.run_node(offline_writer, request)
        finally:
            active_w

### 5.2 先验收契约，再接真实模型

阅读核心函数时抓住四个控制点：计划进入叶子节点；内容任务有预算；不合格稿件不能申请批准；归档只在匹配的人工批准后执行。

下面直接调用实际工程的校验函数，覆盖重复业务 ID、流程版本不兼容和错稿审批摘要。这些错误不能靠网络重试修复。
> 📌 **为什么先测契约，再接真实模型？** 因为这两类失败的性质完全不同：
>
> - **契约错误**（业务 ID 重复、流程版本不兼容、审批摘要不匹配）是**确定性的**——重试一万次还是错；
> - **模型错误**（超时、返回不合规）是**概率性的**——重试往往真的有用。
>
> 如果混在一起调试，你会分不清"这次失败究竟该重试、还是该改代码"。先把确定性错误在门口挡掉，剩下的不确定性才好处理。这就是"快速失败"在 Agent 工程里的具体形态。

In [11]:
test_item = {
    "id": "a",
    "topic": "正常样本",
    "notes": "测试外层编排",
    "required_terms": [],
}
for payload in [
    {"items": [test_item, test_item]},
    {"items": [test_item], "flow_version": "old-version"},
]:
    try:
        Job.model_validate(payload)
    except ValueError:
        pass
    else:
        raise AssertionError("无效任务应被拒绝")
try:
    validate_approval({"approved": True, "digest": "0" * 64}, "1" * 64)
except ValueError:
    print("错稿审批摘要被拒绝")
else:
    raise AssertionError("审批不能绑定错误稿件")
checks["输入身份、版本与审批摘要契约"] = True

错稿审批摘要被拒绝


### 5.3 共用编排的离线分支验收

子进程函数真实执行 `python -m workflow_example.run`，不是伪造状态。离线与 DeepSeek 后端使用同一 `editor`；只替换内容节点。为每个测试使用独立临时目录，结束后清理。
> 📌 **为什么要专门做一个"离线后端"？** 因为这一步要验证的是**业务分支**，不是模型能力。写稿质量天生有抖动，但"不合格就不该申请批准""拒绝了就不该归档"这些规则必须是确定的。把内容生成换成 `offline_writer`，就能用固定输入稳定地走到每一条分支上——`needs_revision`、`rejected`、`archived` 全部可测。
>
> ⚠️ 注意：**恢复时必须使用同一个后端**。同一个 invocation 的前后两个进程如果换了内容来源，回放的旧结果和新的判断就对不上了。工程里靠会话创建时存下的 `backend` / `flow_version` 强制这一点。

In [12]:
async def project_run(
    phase,
    directory,
    *,
    mode="deepseek",
    input_file=None,
    decision="accept",
    fail_after_write=False,
):
    command = [
        sys.executable,
        "-m",
        "workflow_example.run",
        phase,
        "--directory",
        str(directory),
        "--mode",
        mode,
    ]
    if input_file:
        command.extend(["--input", str(input_file)])
    if phase == "approve":
        command.extend(["--decision", decision])
    if fail_after_write:
        command.append("--fail-after-write")
    completed = await asyncio.to_thread(
        subprocess.run,
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
        encoding="utf-8",
        timeout=120,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    if completed.returncode:
        raise RuntimeError(completed.stderr)
    report = json.loads(completed.stdout)
    print(
        phase,
        "→",
        report["status"],
        "; pid =",
        report["pid"],
        "; 模型响应 =",
        report["ledger"]["model_responses"],
        "; 归档回执 =",
        report["ledger"]["receipt_count"],
    )
    return report


for topic, decision, expected in [
    ("失败样本", None, "needs_revision"),
    ("正常样本", "reject", "rejected"),
    ("正常样本", "accept", "archived"),
]:
    with TemporaryDirectory(prefix="adk_ch08_offline_") as folder:
        directory = Path(folder)
        input_file = directory / "job.json"
        input_file.write_text(
            json.dumps(
                {
                    "items": [{**test_item, "topic": topic}],
                    "max_revisions": 0,
                },
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        report = await project_run(
            "start", directory, mode="offline", input_file=input_file
        )
        if decision is not None:
            assert report["status"] == "paused"
            report = await project_run(
                "approve", directory, mode="offline", decision=decision
            )
        assert report["status"] == expected
        assert report["ledger"]["model_responses"] == 0
        assert report["ledger"]["receipt_count"] == int(expected == "archived")
        assert len(report["artifacts"]) == int(expected == "archived")
checks["同一工程的不合格、拒绝与接受分支"] = True

start → needs_revision ; pid = 44456 ; 模型响应 = 0 ; 归档回执 = 0


start → paused ; pid = 44620 ; 模型响应 = 0 ; 归档回执 = 0


approve → rejected ; pid = 3196 ; 模型响应 = 0 ; 归档回执 = 0


start → paused ; pid = 4628 ; 模型响应 = 0 ; 归档回执 = 0


approve → archived ; pid = 38436 ; 模型响应 = 0 ; 归档回执 = 1


### 5.4 进程 A：真实模型并发写作，暂停后退出

输入事实已经按官方文档核对。模型只负责撰写与修改，Python 决定预算、准入和暂停；规则检查只验证长度和术语，完整全文仍需对照事实阅读。

综合案例的正文范围为 60～150 字符，允许比第七章更完整的技术解释。这里最多两项、每项最多修改一次，因此最多四次模型调用步骤。

职责再区分一次：`RequestInput` 发起人工输入请求并使流程暂停；checkpoint 记录可回放的执行结果，支持审批后的恢复。checkpoint 本身不负责审批决定。
> 📌 从**进程**的角度看这一步发生了什么：进程 A 启动 → 建会话 → 跑工作流 → 两个写作节点并发执行 → 走到人工确认 → **暂停** → 进程 A **退出**。
>
> "退出"这两个字很重：进程 A 的内存、局部变量、事件循环**全部消失**。能留下来的只有两样东西——`DatabaseSessionService` 里的会话事件（也就是 checkpoint），以及落盘的 Artifact 与业务 SQLite。
>
> 所以下面这几条断言，验的其实就是"该留的有没有留住"：候选稿在 `pending` 里、状态摘要在会话里、模型调用次数记在业务账本里。

In [13]:
real_job = {
    "flow_version": FLOW_VERSION,
    "max_revisions": 1,
    "items": [
        {
            "id": "dynamic",
            "topic": "ADK Dynamic workflows",
            "required_terms": ["Python", "暂停"],
            "notes": "控制流由 Python 条件、循环和函数调用表达，不等于模型自主委派。"
            "可调用 LLM，也可完全不调用 LLM。人工审核用 RequestInput 暂停，"
            "收到匹配回复后恢复，不用循环轮询。",
        },
        {
            "id": "checkpoint",
            "topic": "ADK checkpoint",
            "required_terms": ["父节点", "子节点"],
            "notes": "恢复时父节点重新进入函数，已完成子节点结果从 checkpoint 回放，"
            "未完成步骤继续。例子是生成草稿后暂停审批，恢复时不重新生成草稿。"
            "调用栈不被冻结；重启需持久化会话和兼容代码；外部幂等另行保证。",
        },
    ],
}
experiment = TemporaryDirectory(prefix="adk_ch08_full_")
directory = Path(experiment.name)
input_file = directory / "job.json"
input_file.write_text(
    json.dumps(real_job, ensure_ascii=False), encoding="utf-8"
)
process_a = await project_run("start", directory, input_file=input_file)
assert process_a["status"] == "paused" and len(process_a["pending"]) == 1
candidates = process_a["pending"][0]["payload"]["articles"]
assert len(candidates) == 2 and all(
    article["passed"] for article in candidates
)
assert 2 <= process_a["ledger"]["model_responses"] <= 4
assert process_a["ledger"]["counts"]["worker_peak"] == 2
assert all(row["tokens"] > 0 for row in process_a["ledger"]["model_usage"])
assert {key for key in process_a["state"] if key.startswith("article:")} == {
    "article:dynamic",
    "article:checkpoint",
}
for candidate in candidates:
    print(json.dumps(candidate, ensure_ascii=False, indent=2))
checks["真实模型、有界并发与独立状态摘要"] = True

start → paused ; pid = 28708 ; 模型响应 = 2 ; 归档回执 = 0
{
  "id": "checkpoint",
  "draft": {
    "title": "ADK checkpoint",
    "body": "ADK checkpoint 在父节点恢复时重新进入函数，已完成子节点结果从检查点回放，未完成步骤继续。例如生成草稿后暂停审批，恢复时不重新生成草稿，调用栈不被冻结。"
  },
  "passed": true,
  "findings": [],
  "revisions": 0
}
{
  "id": "dynamic",
  "draft": {
    "title": "ADK 动态工作流",
    "body": "ADK 动态工作流用 Python 条件、循环和函数调用表达控制流，不等于模型自主委派，可用或不用 LLM。例如写作审核，用 RequestInput 暂停等回复，收到后恢复，而非轮询。"
  },
  "passed": true,
  "findings": [],
  "revisions": 0
}


### 5.5 进程 B：接受候选稿，注入归档完成后的故障

本例以“接受”作为人工决定的测试输入，实际写入本地文件 Artifact 服务。归档叶子禁用局部自动重试，先保存 Artifact 和业务回执，再注入一次连接异常；工作流未完成，但外部写入已经存在。

这个窗口正是单靠 checkpoint 不能保证 exactly-once 的原因。报告必须是 `failed`，不能因为文件已经存在就把整条流程标为成功。
> ⚠️ **这是全章最关键的一个"坏消息"。** 请仔细看这个顺序：
>
> 1. Artifact **已经写进去了**——外部世界变了；
> 2. 业务回执 **已经记上了**——账本也变了；
> 3. 然后节点才抛异常，工作流**没有走完**。
>
> 如果这时系统告诉你"任务失败了，重跑一次吧"，直接重跑会怎样？归档会**被执行第二遍**。**checkpoint 救不了你**：它只知道"这一步没完成"，不知道"外部已经改过了"。
>
> 唯一的出路，是让归档这一步**自己**认得出"这件事我做过了"——也就是下面进程 C 里靠 digest 和唯一操作键做的事。

In [14]:
process_b = await project_run("approve", directory, fail_after_write=True)
assert process_b["status"] == "failed" and process_b["output"] is None
assert [error["code"] for error in process_b["errors"]] == ["ConnectionError"]
assert process_b["ledger"]["receipt_count"] == 1
assert process_b["artifacts"][0]["versions"] == [0]
assert (
    process_b["ledger"]["model_responses"]
    == process_a["ledger"]["model_responses"]
)
print("真实故障：", process_b["errors"])
print("故障后已有归档版本：", process_b["artifacts"][0]["versions"])
checks["真实归档写入后的未完成故障窗口"] = True

approve → failed ; pid = 27068 ; 模型响应 = 2 ; 归档回执 = 1
真实故障： [{'code': 'ConnectionError', 'message': '故障注入：Artifact 和业务回执已保存，节点尚未完成'}]
故障后已有归档版本： [0]


### 5.6 进程 C：恢复同一 invocation，只完成未完成的归档步骤

`recover` 从持久会话读取同一 invocation，使用 `new_message=None` 恢复，不重新提交批准或启动新任务。已经完成的计划、写作与批准结果回放；归档叶子再次执行时，验证已有正文并复用原版本，业务回执以唯一操作键去重。

本实验三个阶段顺序执行，只有一个归档写入方。这证明的是该范围内的恢复幂等；多进程同时发布仍需要外部存储提供原子条件写入或事务协调。候选稿、流程版本与后端必须保持一致。
> 📌 **关键全在"复用"两个字上**：归档叶子**又执行了一遍**（这无法避免，它上次确实没完成），但这一次它先看已有内容——发现正文和自己要写的一模一样，于是**复用原来的版本号**，回执也**没有新增第二条**。
>
> 这才是恢复幂等的真正含义：**不是"不执行"，而是"执行了也不产生第二个结果"。**
>
> 三个进程各自的角色，一句话总结：
>
> - 进程 A：**开个头**，然后带着未完成的现场退出；
> - 进程 B：**试着收尾但失败了**，留下一个"外部已改、账未平"的现场；
> - 进程 C：**回到同一现场收尾**，把重复的那次写入吸收掉。

In [15]:
try:
    process_c = await project_run("recover", directory)
    assert (
        process_c["status"] == "archived" and process_c["output"]["count"] == 2
    )
    assert len({r["pid"] for r in (process_a, process_b, process_c)}) == 3
    assert (
        len({r["invocation_id"] for r in (process_a, process_b, process_c)})
        == 1
    )
    assert process_c["ledger"]["counts"] == {
        "parent": 3,
        "read_plan": 1,
        "article:checkpoint": 1,
        "article:dynamic": 1,
        "worker_peak": 2,
        "archive_attempts": 2,
    }
    assert (
        process_c["ledger"]["model_responses"]
        == process_a["ledger"]["model_responses"]
    )
    assert process_c["ledger"]["receipt_count"] == 1
    assert len(process_c["artifacts"]) == 1 and process_c["artifacts"][0][
        "versions"
    ] == [0]
    assert candidate_digest(candidates) == process_c["output"]["digest"]
    archived_text = process_c["artifacts"][0]["text"]
    assert all(
        article["draft"]["body"] in archived_text for article in candidates
    )
    print(
        "三个进程 PID：", [r["pid"] for r in (process_a, process_b, process_c)]
    )
    print("同一 invocation：", process_c["invocation_id"])
    print("实际执行计数：", process_c["ledger"]["counts"])
    print("实际归档全文：\n", archived_text)
    checks["三进程恢复、模型不重复执行与归档单版本"] = True
finally:
    experiment.cleanup()

recover → archived ; pid = 41972 ; 模型响应 = 2 ; 归档回执 = 1
三个进程 PID： [28708, 27068, 41972]
同一 invocation： e-96a9814c-33c4-45dc-9108-25f15842289c
实际执行计数： {'parent': 3, 'read_plan': 1, 'article:checkpoint': 1, 'worker_peak': 2, 'article:dynamic': 1, 'archive_attempts': 2}
实际归档全文：
 # ADK checkpoint

ADK checkpoint 在父节点恢复时重新进入函数，已完成子节点结果从检查点回放，未完成步骤继续。例如生成草稿后暂停审批，恢复时不重新生成草稿，调用栈不被冻结。

# ADK 动态工作流

ADK 动态工作流用 Python 条件、循环和函数调用表达控制流，不等于模型自主委派，可用或不用 LLM。例如写作审核，用 RequestInput 暂停等回复，收到后恢复，而非轮询。


### 5.7 从实验推广到自己的工程

本例将“编排可靠”与“内容正确”分开。输入事实和全文审核改善内容可靠性；Pydantic、规则检查、预算和存储约束各自保证不同的契约，不应互相替代。

| 观察到的结果 | 证明了什么 | 没有自动证明什么 |
|---|---|---|
| 模型 token 记录 + 完整候选稿 | 真正执行了模型任务 | 所有事实都正确 |
| 三个不同 PID、同一 invocation | 独立进程间恢复 | 任意新版本代码都兼容 |
| 每项工作节点计数保持 1 | 已完成内容步骤没有重跑 | 尚未写入 checkpoint 的模型步骤永不重跑 |
| 两次归档尝试、一个版本与回执 | 指定故障窗口下的业务幂等 | 多发布方自动具备原子写入 |

**这张表是本章的精华**：右边一列比左边一列更重要。工程上真正容易出事的地方，永远是"你以为验过了、其实没验"的那部分。

> 📌 这里还要澄清一个容易混淆的点：**本章的 `checks` 断言和第六章的 `adk eval` 不是一回事。** 本章的 `checks` 统计的是"机制有没有按预期工作"（执行计数、PID、版本号、回执条数），管的是**代码对不对**；`adk eval` 用评估集回放打分，管的是**效果好不好**。两者互补：前者防回归，后者防质量滑坡。

---

## 6. 与 LangGraph 对照 🔄

| 本章机制 | LangGraph 对照 | 差异 |
|---|---|---|
| `RequestInput` + `FunctionResponse` | `interrupt()` + `Command(resume=...)` | 中断与恢复的协议不同，**不能互换** |
| Session checkpoint 回放 | Checkpointer | 持久化配置方式不同 |
| 动态节点调用 `ctx.run_node` | Functional API | 恢复语义不同 |
| 独立业务回执（幂等键） | 没有直接对应物 | 两边都要**自己**保证外部副作用幂等 |

**对照阅读**：建议翻到 [05-持久化-流式与人机协同](../langchain-langgraph/05-持久化-流式与人机协同.ipynb) 对照着看——§1「Checkpointer」对应本章的 checkpoint 回放，§4「人机协同：interrupt 与 Command」对应本章的 `RequestInput`，§5「跨线程长期记忆：Store」对应本章用 `ctx.state` 写的状态摘要。

**两边的机制很像，但恢复协议和持久化配置不能直接互换**——这正是把两条线路都跑一遍的价值。

参考：[Dynamic workflows](https://adk.dev/graphs/dynamic/) · [Human input](https://adk.dev/graphs/human-input/) · [ADK 2.6.3 源码](https://github.com/google/adk-python/tree/v2.6.3/src/google/adk) · [LangGraph Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution)。

---

## 📌 本章要点回顾

- **暂停靠 `RequestInput`，恢复靠匹配的 `FunctionResponse`**（要匹配 function call ID、名称和 invocation），不要用循环轮询；
- **checkpoint 存的是"每一步的答案"，不是"草稿纸"**：恢复时父函数重新进入、从头再跑，已完成的子节点直接回放旧结果。所以父函数必须能重跑，**副作用必须放在独立叶子节点**；
- **三层身份**：`session_id` 定位会话、`invocation_id` 定位一次执行、`run_id` / `node_info.path` 定位执行中的一步；自定义 `run_id` 时，同一个 ID 不能在恢复时代表不同输入；
- **重试预算要显式声明**：不写 `retry_config` 是**不重试**；只写 `RetryConfig()` 却漏写 `max_attempts` 会**静默拿到 5 次**；`max_attempts` **包含**第一次请求；
- **checkpoint 不提供 exactly-once**：外部写入必须靠自己的幂等键去重，"外部已改、节点未完成"这个窗口单靠 checkpoint 补不上；
- **`checks` 管代码对不对，`adk eval` 管效果好不好**，两者互补。

---

## 🧪 本章练习

### 1. 两步人工确认（基础）

在 `RequestInput` 之前插入一个"预审"节点：只有预审通过，才发起人工确认。

**验收**：断言拒绝路径**不产生任何 Artifact**；断言预审节点的执行计数在恢复之后**仍然是 1**（提示：它是已完成子节点，应该走回放）。

### 2. 让审批摘要失效（进阶）

允许人工编辑候选稿正文后重新计算摘要，然后用**旧摘要**提交审批。

**验收**：断言 `validate_approval` 抛错。再用自己的话说明：这个摘要**防住了什么**（回复绑错稿件）、**没防住什么**（回复不是本人发的）。

### 3. 数出框架的默认重试次数（进阶）

写一个"每次必失败"的节点，只写 `RetryConfig()` 而不写 `max_attempts`，统计 `ctx.attempt_count`。

**验收**：打印实际尝试次数并解释为什么是这个数字；再显式设 `RetryConfig(max_attempts=2)` 对比，说明两次实验的差异从哪来。**顺带验证一条**：完全不写 `retry_config` 时，它还会重试吗？

### 4. 模型异常下的部分成功策略（工程）

为其中一项注入模型异常，定义"一项失败是否整批失败"的策略，并写成断言。

**验收**：策略必须**显式**——要么整批失败，要么明确标记部分成功；**禁止静默降级**。并说明为什么"部分成功"必须出现在**返回值**里，而不是只打在日志里。

### 5. 把单写入方换成幂等键服务（综合）

用带 `UNIQUE` 约束的表模拟一个外部幂等键服务，让**两个进程并发**归档同一个 digest。

**验收**：断言最终"一个版本 + 一条回执"；并用文字指出这**仍然不等于**分布式事务——还需要外部存储提供什么能力，才能覆盖所有并发窗口。

参考实现见 [practice/](practice/) 目录下的 `c8_1.py` ~ `c8_5.py`。建议先自己写完再对照。

---

## ✅ 本次实际运行验收

以下验收汇总来自本次完整运行。预期故障以事件数据展示，不是 Notebook 执行错误。

In [16]:
from datetime import datetime, timedelta, timezone

expected = {
    "接受/拒绝、父节点重入与真实 Artifact",
    "顺序变化后的 ID 匹配与子节点回放",
    "写入后故障、真实重试与业务幂等",
    "真实节点超时与明确降级",
    "未声明 max_attempts 时的默认重试预算",
    "输入身份、版本与审批摘要契约",
    "同一工程的不合格、拒绝与接受分支",
    "真实模型、有界并发与独立状态摘要",
    "真实归档写入后的未完成故障窗口",
    "三进程恢复、模型不重复执行与归档单版本",
}
assert set(checks) == expected and all(checks.values())
print(
    "执行时间：",
    datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds"),
)
print(
    "环境：Python",
    sys.version.split()[0],
    "/ google-adk",
    metadata.version("google-adk"),
)
for name in checks:
    print("✅", name)
print("真实 DeepSeek 响应次数：", process_c["ledger"]["model_responses"])
print("服务返回的总 token 使用：", process_c["ledger"]["tokens"])
print("✅ 第八章全部验收通过")

执行时间： 2026-10-14T00:32:57+08:00
环境：Python 3.12.3 / google-adk 2.6.3
✅ 接受/拒绝、父节点重入与真实 Artifact
✅ 顺序变化后的 ID 匹配与子节点回放
✅ 未声明 max_attempts 时的默认重试预算
✅ 写入后故障、真实重试与业务幂等
✅ 真实节点超时与明确降级
✅ 输入身份、版本与审批摘要契约
✅ 同一工程的不合格、拒绝与接受分支
✅ 真实模型、有界并发与独立状态摘要
✅ 真实归档写入后的未完成故障窗口
✅ 三进程恢复、模型不重复执行与归档单版本
真实 DeepSeek 响应次数： 2
服务返回的总 token 使用： 534
✅ 第八章全部验收通过
